# 🕵️ Baseball Lab 2: Stat Detective
Today you're a detective. Your evidence: **every MLB batter's every season, from 1871 to today.**

First a short Python warm-up, so you know exactly what every symbol does. Then the cases.
It takes **under an hour**, and you can stop after any part.

**How it works:**
- Click a gray box and press **Shift + Enter** to run it.
- `___` (three underscores) means **you fill it in**. The box won't work until you do.
- 🔎 **Google it** gives you words to search if you get stuck. Googling is not cheating:
  programmers google things all day long.
- A red error is normal. Read the **last line** of it: that's where Python says what went wrong.

---
## Part 1. Python warm-up 🔥

### `print` shows things on the screen

In [ ]:
print("Home run!")

Three things are happening here:
- `print` is a **command**. It tells Python: show something on the screen.
- The **round brackets** `( )` hold what you give to the command, like handing it the ball.
  A command always needs its brackets, even when they're empty.
- The **quotes** `" "` mean: this is text, show it exactly as it is.

What if we forget the quotes? Run this box. It will show an error, **on purpose**:

In [ ]:
print(Homerun)

Look at the last line of the error: `NameError: name 'Homerun' is not defined`.
Without quotes, Python thinks `Homerun` is the **name** of something it should already know,
and it doesn't know anything called that. So:
**text you want to show → in quotes.**

### Variables: a box with a name on it
A **variable** is like a locker with a name tag. You put something inside, and later you ask for it by its name.
The `=` sign means "put this into the box".

In [ ]:
player = "Aaron Judge"

print(player)
print("player")

Look at the two lines you got:
- `print(player)`, **no quotes**: Python opens the box called `player` and shows what's inside → **Aaron Judge**.
- `print("player")`, **with quotes**: Python just shows the word itself → **player**.

That's why we don't put variable names in quotes: then Python would show the name, not what's in the box.

You can show text and a variable together. Put a **comma** between them:

In [ ]:
player = "Aaron Judge"
number = 99

print("My favorite player is", player, "and he wears number", number)

Notice: `99` has **no quotes**. Numbers don't need them, and Python can do math with them.
Text in quotes is just letters, even if those letters look like numbers. Try it:

In [ ]:
print(100 + 50)
print("100" + "50")

The first line does math: **150**. The second just glues two pieces of text together: **10050**. 😄

### ⚾ Your turn
Change the name and the number to **your** favorite player, then run it.
Remember: the name goes in quotes, the number doesn't.
(The text after `#` is a note for people. Python skips it.)

In [ ]:
player = "Aaron Judge"   # change to your favorite player
number = 99              # change to his number

print("My favorite player is", player, "and he wears number", number)

---
## Part 2. Open the evidence box 📦
The evidence comes from two places: the **Lahman Baseball Database** (a history book of baseball, 1871 to last year)
and **MLB's own live stats** for the newest season. Gluing them together takes a lot of code, so we hid it.
Click the ▶ next to the title and wait about 20 seconds.

It starts with a line you'll see a lot: `import pandas as pd`. That brings in a helper called **pandas**,
which lets Python work with tables, like a spreadsheet. `as pd` gives it a short nickname,
so we can type `pd` instead of `pandas`. (Double-click the title to see all the code.)

In [ ]:
#@title ⚾ Load the MLB data, 1871 to today (click ▶ and wait about 20 seconds)
import datetime
import unicodedata

import pandas as pd
import requests

LAHMAN = "https://raw.githubusercontent.com/daviddalpiaz/pylahman/main/data-raw/"
MLB = "https://statsapi.mlb.com/api/v1/"
STATS = ["G", "AB", "H", "HR", "RBI", "SB", "BB", "SO"]


def plain(text):
    # "Acuña" -> "Acuna": kids type names on an English keyboard
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()


def short(text):
    # "Acuña Jr." -> "acu": the start of a last name, to match players across the two sources
    return plain(text).lower()[:3]


# History: the Lahman Baseball Database (SABR, CC BY-SA 3.0). One row per player, season and team.
batting = pd.read_csv(LAHMAN + "Batting.csv")
people = pd.read_csv(LAHMAN + "People.csv")
teams = pd.read_csv(LAHMAN + "Teams.csv")[["yearID", "teamID", "name"]].rename(columns={"name": "team"})
batting = batting.merge(teams, on=["yearID", "teamID"], how="left")

# A player traded during a season has a row for each team: add them up and keep his last team.
per_season = batting.groupby(["playerID", "yearID"], sort=False)
history = per_season[STATS].sum(min_count=1).join(per_season["team"].last()).reset_index()
people["name"] = people["nameFirst"] + " " + people["nameLast"]
history = history.merge(people[["playerID", "name"]], on="playerID").rename(columns={"yearID": "season"})

# The seasons the history book doesn't have yet come live from MLB (statsapi.mlb.com).
lahman_ids = {
    (int(p.birthYear), int(p.birthMonth), int(p.birthDay), short(p.nameLast)): p.playerID
    for p in people.dropna(subset=["birthYear", "birthMonth", "birthDay"]).itertuples()
}
mlb_names, live = {}, []
try:
    for season in range(int(history["season"].max()) + 1, datetime.date.today().year + 1):
        url = MLB + f"stats?stats=season&group=hitting&season={season}&sportId=1&playerPool=ALL&limit=5000"
        found = requests.get(url, timeout=60).json().get("stats", [])
        hitters = found[0]["splits"] if found else []
        if not hitters:
            continue
        roster = requests.get(MLB + f"sports/1/players?season={season}", timeout=60).json()["people"]
        born = {p["id"]: p.get("birthDate", "") for p in roster}
        for h in hitters:
            p, s = h["player"], h["stat"]
            y, m, d = (born.get(p["id"]) or "0-0-0").split("-")
            player_id = lahman_ids.get((int(y), int(m), int(d), short(p["lastName"])), f"mlb{p['id']}")
            mlb_names[player_id] = p["fullName"]
            live.append({
                "playerID": player_id, "season": season, "team": h.get("team", {}).get("name"), "name": p["fullName"],
                "G": s["gamesPlayed"], "AB": s["atBats"], "H": s["hits"], "HR": s["homeRuns"], "RBI": s["rbi"],
                "SB": s["stolenBases"], "BB": s["baseOnBalls"], "SO": s["strikeOuts"],
            })
except (requests.RequestException, ValueError, KeyError) as problem:
    live = []
    print("MLB's live stats didn't answer, so the newest season is missing today. Everything else works.", problem)

stats = pd.concat([history, pd.DataFrame(live)], ignore_index=True)
# Players active today get the name MLB uses now on their whole career ("Vladimir Guerrero Jr."), without accents.
stats["name"] = stats["playerID"].map(mlb_names).fillna(stats["name"]).map(plain)
# Whole numbers (114, not 114.0); a stat nobody counted back then stays empty: <NA>.
stats[STATS] = stats[STATS].astype("Int64")
stats = stats.sort_values(["playerID", "season"], ignore_index=True)
stats = stats[["name", "season", "team"] + STATS + ["playerID"]]
print("Ready!", len(stats), "rows, seasons", stats["season"].min(), "to", stats["season"].max())

The hidden box made one big table called `stats`. This line shows its first 5 rows, with only the 4 columns we care about.
Python shows the last line of a box by itself, so a table doesn't need `print`.

In [ ]:
stats[["name", "season", "team", "HR"]].head()

That's the top of the table. Two words you'll need:
- A **row** is one line across: **one player in one season**.
- A **column** is one kind of number going down, like `HR`.

### Square brackets: pick what you want from the table
**Why we need them:** this table has **12 columns**, and most of the time we only care about a few.
So we tell Python which ones to show.

To get **one** column, put its name in quotes inside square brackets:

```
stats["HR"]
```

To get **several** columns, you see double brackets:

```
stats[["season", "HR"]]
```

Double brackets look like a new symbol, but they aren't. They are **two different things**, one inside the other:

```
stats[ ["season", "HR"] ]
```

- The **outside** `[ ]` means "pick from the table", same as before.
- The **inside** `[ ]` is a **list**: a few things in a row, with commas between them, like a lineup card.
  In other programming languages it's called an **array**.

The list is how we hand Python **many** column names as **one** thing.
You'll build your own lists in Lab 4. Today you only need to know what it's doing here.

To see it, let's put a list into a box of its own. Run this:

In [ ]:
my_columns = ["season", "team", "HR"]

print(my_columns)
stats[my_columns].head(3)

Look at what happened:
- `print(my_columns)` shows the list itself: three names, in order.
- `stats[my_columns]` shows the table with only those three columns.
- `.head(3)` shows only the first 3 rows.

So `stats[my_columns]` does exactly the same thing as `stats[["season", "team", "HR"]]`.
That's where the double brackets come from: a list, sitting inside the "pick from the table" brackets.

**What if we leave out the inside brackets?** Run this box. It will show an error, **on purpose**:

In [ ]:
stats["season", "HR"]

The last line says `KeyError: ('season', 'HR')`. `KeyError` means **"I can't find that name in the table."**
Without the list, Python looks for **one** column with that strange double name, and there isn't one.
So: **one column → single brackets. Several columns → a list inside the brackets.**

The columns we'll use today:

| column | what it means |
|---|---|
| `name` | the player's name |
| `season` | the year |
| `team` | his team that season |
| `G` | games played |
| `H` | hits |
| `HR` | home runs |
| `RBI` | runs batted in |
| `SB` | stolen bases |
| `BB` | walks |
| `SO` | strikeouts |

### ⚾ Your turn
How many rows are in the box? Python has a command that counts things.
It's 3 letters long and short for "length". Write it in place of `___`.

🔎 **Google it:** `python how to count items in a list`

In [ ]:
print("Rows in the box:", ___(stats))

Every row is one player in one season. That's a lot of seasons!

---
## Part 3. Track down your player 🎯
Right now `stats` has **every** player in it. If you asked for the biggest number, you'd get the biggest number
of **anybody**, not your player. So first we need a smaller table with **only your player's rows**.

To do that, you need to know the difference between two symbols that look almost the same:

| symbol | what it does | read it as |
|---|---|---|
| `=` | **puts** something into a box | "put this into the box" |
| `==` | **asks** a question | "are these the same?" |

Run this and watch what each one does:

In [ ]:
team = "Yankees"           # one =  puts Yankees into the box

print(team == "Yankees")   # two == asks: is it the same as "Yankees"?
print(team == "Dodgers")   # two == asks: is it the same as "Dodgers"?

- `team = "Yankees"` gives an order: put **Yankees** into the box. Nothing shows on the screen.
- `team == "Yankees"` asks a question. Python looks in the box and answers `True`, which means **yes**.
- `team == "Dodgers"` asks again. The answer is `False`, which means **no**.

`True` and `False` are Python's two answers. They have capital letters and no quotes. You'll use them a lot in Lab 3.

Asking a question **doesn't change the box**. After those two questions, `team` still holds Yankees.

### Step 1: ask every row the question
Here is the cool part. When you ask a question about a **column**, Python asks it about **every row, all at once**:

In [ ]:
name = "Aaron Judge"

answers = stats["name"] == name

print("Rows asked:", len(answers))
answers.head()

The line `answers = stats["name"] == name` has **both** symbols. Python reads it in this order:
1. **Right side first:** for every row, is the name the same as `name`?
2. **Then the left side:** put all those answers into a box called `answers`.

The number on the left of each line is the row number, and on the right is the answer for that row.
The table's first rows are other players, so they say `False`. Further down, the rows with your player say `True`:

| row | name in that row | `== "Aaron Judge"`? |
|---|---|---|
| first row | David Aardsma | `False` |
| many rows later | Aaron Judge | `True` |
| many rows later | Shohei Ohtani | `False` |

(The `Name: ... dtype: bool` line at the bottom is just a label from pandas. You can skip it.)

### Step 2: keep the rows that say `True`
Now hand those answers to the table, inside square brackets:

In [ ]:
seasons = stats[answers]

print("Rows that said True:", len(seasons))
seasons[["name", "season", "team"]].head()

It works like cutting a roster: rows that say `True` make the team, rows that say `False` are cut.
Now `seasons` is a much smaller table, and every row in it is your player.

Square brackets look at **what you put inside them**:
- column names → pick those **columns**
- `True`s and `False`s → pick those **rows**

### The short way: both steps on one line
Programmers skip the `answers` box and put the question straight inside the brackets:

```
seasons = stats[stats["name"] == name]
```

It looks scary, but it's the same two steps. Read it **from the inside out**:

| order | this part | what it does |
|---|---|---|
| 1 | `stats["name"]` | take the column of names |
| 2 | `== name` | ask every row: is this my player? |
| 3 | `stats[ ... ]` | keep only the rows that said `True` |
| 4 | `seasons =` | put that smaller table into a box called `seasons` |

That's why `stats` is written twice: the inside one is where we **ask**, the outside one is where we **keep rows**.
These brackets are **not** a list. There are no commas, just a question inside.

**What if we use one `=` by mistake?** Run this box. It will show an error, **on purpose**:

In [ ]:
seasons = stats[stats["name"] = name]

Read the last line: Python even says `Maybe you meant '==' instead of '='?`
A `SyntaxError` means Python can't understand how the line is written. In the middle of those brackets it needs a **question**,
and one `=` is an order to put something in a box. This is one of the most common mistakes in Python, even for grown-up programmers.

### Now do it with your player
The box below uses the short way. Its last line uses the list from Part 2: `seasons[[...]]` picks which 7 columns to show, out of 12.

In [ ]:
name = "Aaron Judge"   # change to your favorite player, in quotes, spelled exactly right

seasons = stats[stats["name"] == name]

print(name, "has", len(seasons), "rows: one for each season")
seasons[["season", "team", "G", "H", "HR", "RBI", "SB"]]

If it says **0 rows**, check the spelling: `"Aaron Judge"` works, `"aaron judge"` doesn't.
Capital letters matter to Python.

Today's players are written the way MLB writes them: `"Vladimir Guerrero Jr."`, `"Bobby Witt Jr."`.
All names are **without accent marks**, so type `"Julio Rodriguez"`, not `"Julio Rodríguez"`.

### Two things in the next box
- `stat = "HR"` puts a column name into a box. So `seasons[stat]`, with **no quotes**, opens the box and
  means the same as `seasons["HR"]`. **Why bother?** Now, to look at a different stat, you change only one line.
- The **dot** in `seasons[stat].max()` means "do this to that": take this column, and run `max` on it.
  You've seen it already in `.head()`.

### ⚾ Your turn: his best season ever
Every column has a command that finds its **biggest** number. It's 3 letters, the opposite of "min".
Write it in place of `___`. The brackets `()` after it stay.

If your player is fast rather than strong, change `"HR"` to `"SB"` (stolen bases) or `"H"` (hits) first.

🔎 **Google it:** `pandas biggest value in a column`

In [ ]:
stat = "HR"

print("Best season:", seasons[stat].___())

And here is how many he gets in a **normal** season. `mean` means "average": add up all his seasons
and share them out equally. `round( )` cuts off the decimals. Just run it:

In [ ]:
print("Average season:", round(seasons[stat].mean()))

See `<NA>` and a red `TypeError`? `<NA>` means **empty**: Python found no seasons, so the name in Part 3
is spelled differently. Fix it there, run that box again, then come back.

---
## Part 4. The biggest seasons in history 💥
`sort_values` puts all the rows in order. `ascending=False` means **biggest first**
(without it, the smallest would come first). `.head(5)` shows only the top 5.

**Wait, one `=` again?** Inside a command's round brackets, `=` chooses a **setting**, like flipping a switch on the command.
*Ascending* means "going up", from small to big. So `ascending=False` says: "switch going up **off**."
It's not a box and it's not a question.

Look at the last line too. The list has `stat` in it with **no quotes**, so Python opens the box and uses the column name inside.
That's why the table always shows the stat you picked.

In [ ]:
stat = "HR"

biggest = stats.sort_values(stat, ascending=False)
biggest[["name", "season", "team", stat]].head(5)

Who's #1? He also has the most home runs in a whole career. 👑

### ⚾ Your turn
Change **only the first line**: replace `"HR"` with `"H"` (keep the quotes).
Run it and see the most hits anyone ever got in one season.

Then try `"RBI"` and `"SO"` in the same spot.

In [ ]:
stat = "HR"   # change this word

biggest = stats.sort_values(stat, ascending=False)
biggest[["name", "season", "team", stat]].head(5)

---
## Part 5. Who's better? ⚔️
We built you a machine called `compare`. You give it two players and a stat,
and it adds up their **whole careers** and tells you who wins.

**Why a machine?** So you don't have to write all these lines again for every argument. You write them once, then use one short line.

Here's what's inside, in plain words. You don't need to know every detail yet, and in Lab 3 you'll take it apart yourself.
- `def compare(player1, player2, stat):` teaches Python a new command called `compare`. `player1`, `player2` and `stat`
  are empty boxes that get filled when you use the machine.
- `total1 = ...` is your trick from Part 3: keep only one player's rows, then take the `stat` column.
  `.sum()` adds up everything in that column.
- `print` shows both totals.
- `if` and `else` decide who wins: if total 1 is bigger, player 1 wins, otherwise player 2 does.

Run this box once so Python learns the machine (nothing shows up yet, that's normal):

In [ ]:
def compare(player1, player2, stat):
    total1 = stats[stats["name"] == player1][stat].sum()
    total2 = stats[stats["name"] == player2][stat].sum()
    print(player1, ":", total1)
    print(player2, ":", total2)
    if total1 > total2:
        print("Winner:", player1, "🏆")
    else:
        print("Winner:", player2, "🏆")

Now use it. Inside the brackets: first player, second player, the stat. Each one in quotes, with commas between them.

In [ ]:
compare("Aaron Judge", "Shohei Ohtani", "HR")

### ⚾ Your turn: settle an argument
Pick two players you'd argue about and a stat from the table in Part 2.
Replace the names and the stat below, keeping the quotes and the commas.

**Careful with spelling:** if a total says **0** and you didn't expect it, check the name. The machine found no rows for that name, so the total is 0, and the other player "wins" with no warning. A `KeyError` means the stat's name is spelled wrong.

In [ ]:
compare("Mookie Betts", "Juan Soto", "H")   # change the names and the stat

---
### 🏆 Case closed, Detective!
Today you learned:
- `print( )` shows things, and **quotes** mean "this is text"
- a **variable** is a box with a name: `player = "..."`
- `=` puts into a box, `==` asks "are these the same?", and Python answers `True` or `False`
- how to read `stats[stats["name"] == name]` from the inside out
- `[[ ]]` is a **list** inside the "pick from the table" brackets (more about lists in Lab 4)
- `len( )` counts, `.max()` finds the biggest, `.mean()` finds the average
- and you used a machine someone else built: `compare( )`

Next time: we'll look inside that machine and teach Python to make decisions, like an umpire.